In [ ]:
import sys
import os
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath('..'))

from DataSet.DataSetCombiner import DataSetCombiner

In [ ]:
combiner = DataSetCombiner()
merged_df = combiner.execute_pipeline()

if not merged_df.empty:
    display(merged_df)
else:
    print("The dataframe is empty. Please check the logs above for errors.")

In [ ]:
merged_df.dropna(inplace=True)

In [ ]:
merged_df

In [ ]:
merged_df.describe()

In [ ]:
merged_df.dropna(inplace=True)
merged_df.drop_duplicates(inplace=True)

In [ ]:
merged_df

In [ ]:
merged_df.columns

In [ ]:
skill_cols = [f'Skill_{i}' for i in range(1, 11)]

plt.figure(figsize=(16, 8))
for i, col in enumerate(skill_cols, 1):
    plt.subplot(2, 5, i)
    sns.histplot(merged_df[col], bins=20, kde=False, color='skyblue')
    plt.title(col)
plt.tight_layout()
plt.show()

impact_mapping = {'Low': 0, 'Moderate': 1, 'High': 2} 

merged_df['AI_Impact_Encoded'] = merged_df['AI Impact Level'].map(impact_mapping)

corr_matrix = merged_df[skill_cols + ['AI_Impact_Encoded']].corr()

plt.figure(figsize=(6, 6))
sns.heatmap(
    corr_matrix[['AI_Impact_Encoded']].drop('AI_Impact_Encoded'), 
    annot=True, 
    cmap='coolwarm', 
    vmin=-1, 
    vmax=1
)

plt.title('Correlation of Skills on AI Impact Level')
plt.show()

merged_df = merged_df.drop(columns=['AI_Impact_Encoded'])

## Dropping columns from skill_1 to skill_10 because The correlation with AI Impact Level for every single one of those skills is practically 0.00 (ranging from -0.01 to 0.01).

In [ ]:
skill_cols = [f'Skill_{i}' for i in range(1, 11)]
if set(skill_cols).issubset(merged_df.columns):
    merged_df = merged_df.drop(columns=skill_cols)

In [ ]:
merged_df

In [ ]:
merged_df.columns = (
    merged_df.columns
    .str.replace(r'\s*\(\%\)', '', regex=True)  
    .str.replace(' ', '_')                      
)

merged_df

In [ ]:
columns_to_drop = [
    'Average_Salary',   
    'Years_Experience',
    'Education_Level',  
    'Risk_Category_y'
]
merged_df = merged_df.drop(columns=columns_to_drop, errors='ignore')
merged_df = merged_df.rename(columns={'Risk_Category_x': 'Risk_Category'})
merged_df

In [ ]:
merged_df['Job_Title'].value_counts()

In [ ]:

sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

sns.countplot(data=merged_df, x='AI_Impact_Level', order=['High', 'Moderate', 'Low'], 
              palette='magma', ax=axes[0])
axes[0].set_title('Distribution of AI Impact Level')
axes[0].set_ylabel('Number of Jobs')
axes[0].set_xlabel('Impact Level')

sns.countplot(data=merged_df, y='Industry', order=merged_df['Industry'].value_counts().index, 
              palette='magma', ax=axes[1])
axes[1].set_title('Job Count by Industry')
axes[1].set_ylabel('Industry')
axes[1].set_xlabel('Number of Jobs')

sns.histplot(data=merged_df, x='Median_Salary_(USD)', bins=30, kde=True, 
             ax=axes[2], color='#B12A90') 
axes[2].set_title('Distribution of Median Salary (USD)')
axes[2].set_xlabel('Salary (USD)')
axes[2].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 18))

sns.countplot(data=merged_df, y='Industry', hue='AI_Impact_Level', 
              hue_order=['High', 'Moderate', 'Low'], palette='magma', ax=axes[0])
axes[0].set_title('AI Impact Level on Different Industries', fontsize=14)
axes[0].set_xlabel('Number of Jobs')
axes[0].set_ylabel('Industry')

sns.boxplot(data=merged_df, x='AI_Impact_Level', y='Median_Salary_(USD)', 
            order=['High', 'Moderate', 'Low'], palette='magma', ax=axes[1])
axes[1].set_title('Median Salary vs AI Impact Level', fontsize=14)
axes[1].set_xlabel('AI Impact Level')
axes[1].set_ylabel('Median Salary (USD)')

sns.countplot(data=merged_df, y='Required_Education', hue='AI_Impact_Level', 
              hue_order=['High', 'Moderate', 'Low'], palette='magma', ax=axes[2])
axes[2].set_title('AI Impact Level on Education Levels', fontsize=14)
axes[2].set_xlabel('Number of Jobs')
axes[2].set_ylabel('Required Education')

plt.tight_layout()
plt.show()

In [ ]:
merged_df.to_csv('../DataSet/cleaned_job_data.csv', index=False)
print("Data successfully saved to DataSet/cleaned_job_data.csv!")